# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My Lane as aan ML task would be Scoring, Sharpened

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target or proxy

Target: predicted CTR, based on position tier and other page attributes known before the decision point
Proxy label for ranking: gap = actual CTR − predicted CTR (the page-level `ctr` column, pulled from Search Console-style aggregation in the warehouse — impressions and clicks over the trailing 90 days).

Worth being upfront about this: it's a proxy, not a ground-truth label. `ctr` is a real, observed outcome — it actually happened. But "should be flagged for review" isn't something that exists anywhere in the data. No reviewer has ever gone through and labeled a page as needing a rewrite. I'm defining that signal myself, as underperformance relative to expectation. That's a modeling choice, not a fact — and it's the first thing I'll revisit if the top of the queue doesn't hold up on inspection (see the signal audit in week 6).

## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success metric

Primary (model-quality) metric: MAE of predicted CTR vs. actual CTR, evaluated on clients held out of training The split is at the client level, not the row level, so no single client's pages end up split across train and test (this mirrors the logic in the reference pipeline). I'm benchmarking against a naive tier-mean baseline: if the model can't beat "just guess the tier average," it isn't actually adding value.

Deployment metric, once it's available: precision@K Of the top K pages the queue surfaces each week, how many does a reviewer actually agree are real gaps? That's the metric the business actually cares about but it depends on human-labeled review outcomes that don't exist yet, so it's a week-6-and-beyond addition, not something I can honestly compute today. MAE is what I can defend right now, with the data I actually have.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [3]:
import pandas as pd

url = 'https://raw.githubusercontent.com/PrathamDudani/FlyRank_Assignment/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(url)
visible = df[(df['avg_position'] > 0) & (df['impressions_90d'] >= 500)].copy()

cols = ['content_id', 'client_id', 'position_tier', 'avg_position', 'main_intent',
        'impressions_90d', 'clicks_90d', 'ctr', 'word_count', 'freshness_tier']
print(f"{len(visible):,} rows, {visible['client_id'].nunique()} clients -- one row per content_id")
visible[cols].head(8)


16,726 rows, 28 clients -- one row per content_id


,content_id,client_id,position_tier,avg_position,main_intent,impressions_90d,clicks_90d,ctr,word_count,freshness_tier
0,content_304f48230142,client_f369cb89fc,striking,10.6,transactional,3803,29,0.76,3221.0,0-30
1,content_a1fb4e703a9e,client_4e07408562,page_3_5,20.3,informational,15320,7,0.05,2481.0,0-30
2,content_9aa793d4d895,client_7f2253d7e2,page_3_5,36.5,informational,12581,11,0.09,3515.0,0-30
3,content_331d6c4de07b,client_19581e27de,page_1,6.2,commercial,11751,58,0.49,NaN,0-30
4,content_d99b7a2d90ca,client_3fdba35f04,page_3_5,44.0,informational,19140,24,0.13,2803.0,0-30
5,content_d4084a4bc775,client_f369cb89fc,page_1,8.5,transactional,3970,1,0.03,3080.0,0-30
7,content_a63219c6e95a,client_19581e27de,page_3_5,21.2,commercial,1724,1,0.06,NaN,0-30
8,content_5e6c160719bc,client_6208ef0f77,page_3_5,46.0,informational,32574,29,0.09,3807.0,0-30


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Why ML beats a fixed rule here

A fixed rule here would mean manually nesting categorical splits — "predict CTR as the average for this tier, this intent, this freshness bucket" — and just looking the answer up in a table. I tested how far that actually gets you: going from a tier-only lookup to a tier-*and*-intent lookup barely moves the error, and the leftover spread *inside* each bucket barely shrinks either (numbers below). Two things follow from that:

1. Most of the real variation in CTR isn't explained by any single categorical split you'd think to hand-write — it's spread across several weaker, continuous signals (word count, content age, freshness) that only matter when combined.
2. Manually crossing every categorical column (tier × intent × content type × freshness × word count tier) to chase that variation just creates dozens of tiny buckets, many too small to trust — the classic problem with hand-built lookup tables.

That's the case for a model: something that can weigh several continuous features and their interactions at once, rather than a person pre-deciding which combinations of categories deserve their own row in a table.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.